In [2]:
from nemo.collections.asr.parts.submodules.wfst_decoder import RivaGpuWfstDecoder

from inference_funcs import load_bit_phoneme_model, load_gru, evaluate_model
from dataset import getDatasetLoaders
import numpy as np
import torch
import torch.nn.functional as F

In [3]:
load_predictions = True

device = 'cuda'

model_filepath = "/data/models/time_masked_transfomer_characters_phonemes_80ms_seed_0/"
gru = False

if load_predictions:
    
    logits_torch_arranged = torch.load(f"{model_filepath}logits_arranged.pth").to(dtype=torch.float32, device=device)
    log_probs_arranged = torch.load(f"{model_filepath}log_probs_arranged.pth").to(dtype=torch.float32, device=device)
    log_probs_length = torch.load(f"{model_filepath}log_probs_length.pth").to(dtype=torch.int64, device='cpu')

else: 
    
    if gru:
        model, args = load_gru(model_filepath)
        
    else:    
        model, args = load_bit_phoneme_model(model_filepath)
        
    model = model.to(device)

    data_file = '/data/neural_data/ptDecoder_ctc_both_char_phoneme'
    trainLoaders, testLoaders, loadedData = getDatasetLoaders(
            data_file, 8, None, 
            False
        )

    outputs, cer, per_day_cer, _, _ = evaluate_model(model, loadedData, args, partition='test', device='cuda', verbose=False)

    num_classes = 41
    if gru:
        add_length = 1
    else:
        add_length = 0

    logits = np.zeros((len(outputs['logits2']), max(outputs['logitLengths2'])+add_length, num_classes))
    for idx, l in enumerate(outputs['logits2']):
        l_length = outputs['logitLengths2'][idx]
        if gru:
            l_length += add_length
        logits[idx, :l_length, :] = l
        
    logits_torch = torch.from_numpy(logits)
    logits_torch_blank_last = torch.concat((logits_torch[:, :, 1:], logits_torch[:, :, 0:1]), dim=-1) # move blank to end
    logits_torch_arranged = torch.concat((logits_torch_blank_last[:, :, -1:], logits_torch_blank_last[:, :, -2:-1], logits_torch_blank_last[:, :, :-2]), dim=-1)
    
    log_probs_arranged = F.log_softmax(logits_torch_arranged, dim=-1).to(dtype=torch.float32, device=device)
    log_probs_length = torch.from_numpy(np.array(outputs['logitLengths2'])).to(dtype=torch.int64, device='cpu')

    torch.save(logits_torch_arranged, f"{model_filepath}/logits_arranged.pth")
    torch.save(log_probs_arranged, f"{model_filepath}log_probs_arranged.pth")
    torch.save(log_probs_length, f"{model_filepath}log_probs_length.pth")


In [5]:

language_model_fst_path = "/data/code/nejm-brain-to-text/language_model/pretrained_language_models/openwebtext_1gram_lm_sil/TLG_opt_with_symbols.fst"
language_model_path_3g = '/data/lm/TLG_opt_with_symbols.fst'

max_mem = 50000000
blank_penalty = 0.7
lm_weight = 1.0
beam_size = 18
nbest_size = 1
batch_size = 440
acoustic_scale = 10

decoder = RivaGpuWfstDecoder(lm_fst=language_model_path_3g, decoding_mode="nbest", 
                             beam_size=beam_size, lm_weight=lm_weight,
                             nbest_size=nbest_size, max_mem=max_mem, blank_penalty=blank_penalty, 
                             max_batch_size = batch_size, acoustic_scale = acoustic_scale)

In [6]:
T = 3.0
log_probs_arranged = F.log_softmax(logits_torch_arranged/T, dim=-1).to(dtype=torch.float32, device=device)

In [7]:
import pickle
with open('/data/text/validation_sentences_ground_truth.pkl', 'rb') as f:
    val_ground_truth_all = pickle.load(f)

In [8]:
log_probs_arranged.shape

torch.Size([880, 230, 41])

In [9]:
log_probs_one_sample = torch.unsqueeze(log_probs_arranged[400], dim=0)
log_probs_length_one_sample = torch.unsqueeze(log_probs_length[400], dim=0)
print(log_probs_length_one_sample)
print(log_probs_one_sample.shape)
print(log_probs_length_one_sample)

tensor([61])
torch.Size([1, 230, 41])
tensor([61])


In [ ]:
hypotheses = decoder._decode_nbest(log_probs_arranged, log_probs_length)

In [21]:
decoded_sentences = []
for i in range(880):
    words_tuple = hypotheses[i]._hypotheses[0].words
    decoded_sentences.append(' '.join(words_tuple).lower())
    
from cer_wer import  _cer_and_wer
_, wer, _ =  _cer_and_wer(decodedSentences=decoded_sentences, trueSentences=val_ground_truth_all)
print(wer)

WfstNbestUnit(words=('FRIDAY', 'AFTERNOON', 'AT', 'FIVE', 'THIRTY'), timesteps=(0, 0, 19, 31, 41), alignment=(1, 0, 0, 0, 0, 0, 0, 0, 15, 29, 7, 0, 10, 10, 10, 19, 19, 1, 1, 0, 3, 15, 15, 32, 13, 13, 24, 35, 24, 0, 1, 1, 3, 3, 32, 32, 0, 1, 15, 15, 7, 7, 36, 36, 0, 0, 1, 1, 33, 33, 13, 0, 10, 10, 19, 19, 0, 0, 0, 1, 1), score=78843.03125)
WfstNbestUnit(words=('VERY', 'OFTEN', 'AT', 'FIVE', 'THIRTY'), timesteps=(0, 7, 7, 7, 7), alignment=(0, 0, 0, 0, 0, 0, 0, 0, 36, 0, 12, 0, 29, 0, 19, 19, 19, 1, 1, 5, 15, 15, 15, 32, 32, 32, 32, 4, 24, 0, 1, 1, 3, 3, 32, 32, 0, 1, 15, 15, 7, 7, 36, 36, 0, 0, 1, 1, 33, 33, 13, 0, 10, 10, 19, 19, 0, 0, 0, 1, 1), score=79666.234375)
WfstNbestUnit(words=('VERY', 'AFTERNOON', 'AT', 'FIVE', 'THIRTY'), timesteps=(0, 7, 7, 7, 7), alignment=(0, 0, 0, 0, 0, 0, 0, 0, 36, 0, 12, 0, 29, 0, 19, 19, 19, 1, 1, 0, 3, 15, 15, 32, 13, 13, 24, 35, 24, 0, 1, 1, 3, 3, 32, 32, 0, 1, 15, 15, 7, 7, 36, 36, 0, 0, 1, 1, 33, 33, 13, 0, 10, 10, 19, 19, 0, 0, 0, 1, 1), score=80260

0.23822513184215313
